# 🏭 DL & GenAI Project (T3-2026) — Milestone 5 notebook
Roll no. `23f3001567` · W&B project `23f3001567-t32026`

**Milestone 5 = turn trained segmentation models into a final, competition-ready submission.**
Raw image → model(s) → *(TTA, un-flipped)* → probabilities → *(ensemble)* → threshold → connected-component / minimum-area filter → RLE → `submission.csv`.

## How to use
Kaggle → add the competition data → **Add Input → Notebooks → Your work → attach the notebook(s) that produced your `*_best.pth` files** (effb0 / effb2 / effb3 / effb2b / resnet34 / scratch …) → GPU **T4** → *Run All*.
* If **no checkpoint** is found, the notebook trains two small from-scratch U-Nets (~10 min) so that it still runs end-to-end (set `FALLBACK_TRAIN = False` to forbid that).
* Everything is **streamed** (never all test images / all probabilities in memory), so it fits Kaggle's memory and time limits (Section 9 measures this).
* Every section prints lines starting with **`ANSWER Qn`**; the last cell collects them in an **ANSWER SHEET** (also saved to `milestone5_answers.txt/json`).

| Form question | Topic | Section |
|---|---|---|
| Q1 | logits, probability thresholds, strict-inequality trap | 3 |
| Q2 | per-class threshold / min-area selection from a sweep table | 4 |
| Q3 | connected components, connectivity, area filtering | 5 |
| Q4 | does post-processing improve the official score? (ablation ladder) | 6 |
| Q5 | recall ceiling of min-area filtering at the pair level | 7 |
| Q6 | TTA: flip axes and the "forgot to un-flip" bug | 8 |
| Q7 | ensembling: soft averaging, hard voting, where to average | 9 |
| Q8 | inference budget: memory and time of two ensemble configurations | 10 |
| Q9 | final inference notebook, submission integrity, experiment discipline | 11 – 12 |

> ⚠️ The question bodies of the Google Form are images, so this notebook computes **all the quantities those topics can ask for** (tables + `ANSWER Qn` lines) on **your own data and models**. Copy the numbers that match the wording of each question. If a question uses toy numbers (e.g. a given sweep table), the helper functions (`class_gain`, `apply_filter`, `budget`, …) can be called directly on them.

## 0. Settings

In [ ]:
# ===================== SETTINGS (the only cell you normally edit) =====================
ROLL_NO        = "23f3001567"
WANDB_PROJECT  = f"{ROLL_NO}-t32026"
INPUT_ROOT, OUT_DIR = "/kaggle/input", "/kaggle/working"
QUICK_TEST     = False          # True = tiny dry run (few images, 1 epoch) just to check that the code runs; answers are then meaningless

SEED, VAL_FRACTION = 42, 0.2    # IDENTICAL to the project notebook -> the same validation images as when your checkpoints were trained
VAL_MAX        = 1200           # validation images used for sweeps (None = all ~2,279). 1,200 images ~ 1 GB of fp16 probabilities per model-variant
MAX_MEMBERS    = 4              # at most this many checkpoints enter the ensemble (best ones by validation score)
INFER_BATCH    = 32
THR_GRID  = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.60, 0.70, 0.80, 0.90]   # probability thresholds
AREA_GRID = [0, 10, 20, 40, 60, 100, 150, 200, 300]                                                # minimum area (pixels)
GATE_GRID = [0.0, 0.5, 0.7, 0.9]                                                                   # confidence gate (0.0 = off)

FALLBACK_TRAIN = True           # train small from-scratch U-Nets if fewer than 2 checkpoints are found
FALLBACK_IMAGES, FALLBACK_EPOCHS, FALLBACK_SEEDS = 3000, 8, [42, 7]
USE_WANDB      = True

TIME_LIMIT_MIN, GPU_MEM_LIMIT_MB, RAM_LIMIT_GB = 540, 15000, 29   # Kaggle: ~9 h session budget used here, T4 = 16 GB, RAM = 29 GB -> change if the question states other limits
if QUICK_TEST: VAL_MAX, FALLBACK_IMAGES, FALLBACK_EPOCHS = 160, 320, 1

import os, time, random, json, math, copy, itertools, shutil, hashlib, warnings
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

H, W    = 128, 800
HW      = H * W                                  # 102,400 pixels per image
CLASSES = ["defect_a", "defect_b", "defect_c", "defect_d"]
NC      = len(CLASSES)
ANS     = {}                                     # formatted answers are collected here
os.makedirs(OUT_DIR, exist_ok=True)
print("settings ok | quick test:", QUICK_TEST)

## 1. Data, RLE and the official metric
Same conventions as Milestones 1–4: RLE is **1-based, column-major**; empty mask = single space `" "` in the submission; the validation split is the **project's split** (seed 42, stratified on the defect pattern).

In [ ]:
IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")
def find_file(root, names):
    for dirpath, _, files in os.walk(root):
        for n in names:
            if n in files: return os.path.join(dirpath, n)
def find_image_folder(root, split):
    best = None
    for dirpath, _, files in os.walk(root):
        if os.path.basename(dirpath) == split:
            n = sum(f.lower().endswith(IMG_EXT) for f in files)
            if n and (best is None or n > best[1]): best = (dirpath, n)
    return best[0] if best else None
def index_images(folder):
    return {os.path.splitext(f)[0]: os.path.join(folder, f) for f in os.listdir(folder) if f.lower().endswith(IMG_EXT)}

TRAIN_CSV  = find_file(INPUT_ROOT, ["train.csv", "train_1.csv"])
SAMPLE_CSV = find_file(INPUT_ROOT, ["sample_submission.csv", "sample_submission_1.csv"])
train_dir, test_dir = find_image_folder(INPUT_ROOT, "train"), find_image_folder(INPUT_ROOT, "test")
assert TRAIN_CSV and SAMPLE_CSV and train_dir and test_dir, "Data not found - did you add the competition via 'Add Input'?"
TRAIN_PATHS, TEST_PATHS = index_images(train_dir), index_images(test_dir)

def rle_to_mask(rle, shape=(H, W)):
    flat = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    if isinstance(rle, str) and rle.strip():
        nums = np.array(rle.split(), dtype=np.int64)
        for s, l in zip(nums[0::2] - 1, nums[1::2]):
            a, b = max(int(s), 0), min(int(s + l), shape[0] * shape[1])
            if b > a: flat[a:b] = 1
    return flat.reshape(shape, order="F")

def mask_to_rle(mask):
    pixels = np.concatenate([[0], np.asarray(mask, dtype=np.uint8).flatten(order="F"), [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

train_df  = pd.read_csv(TRAIN_CSV)
sample_df = pd.read_csv(SAMPLE_CSV, keep_default_na=False)
rle_pivot = train_df.pivot(index="image_id", columns="defect_type", values="mask_rle")[CLASSES].fillna("")
if QUICK_TEST: rle_pivot = rle_pivot.sample(min(600, len(rle_pivot)), random_state=SEED)
has       = (rle_pivot != "").astype(int)
rle_table = {i: list(r) for i, r in zip(rle_pivot.index, rle_pivot.values)}
all_ids   = list(has.index)
real = train_df.mask_rle.dropna()
assert all(mask_to_rle(rle_to_mask(r)) == r for r in real.sample(min(500, len(real)), random_state=SEED)), "RLE round-trip failed"
print("✅ RLE encode/decode round-trip exact |", len(all_ids), "train images |", len(TEST_PATHS), "test images |",
      f"{100 * (has.sum(axis=1) == 0).mean():.1f}% of train images fully clean")

# ---- the project's stratified split (same code as the v5 notebook) ----
from sklearn.model_selection import train_test_split
pattern = has.astype(str).agg("".join, axis=1)
pattern = pattern.where(pattern.map(pattern.value_counts()) >= 3, "rare")
train_ids, val_ids = train_test_split(all_ids, test_size=VAL_FRACTION, stratify=pattern.loc[all_ids], random_state=SEED)
val_ids_eval = list(val_ids[:VAL_MAX]) if VAL_MAX else list(val_ids)
N = len(val_ids_eval)
print(f"train {len(train_ids)} | validation {len(val_ids)} | used for sweeps: {N}")

In [ ]:
def load_gray(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    return img if img.shape == (H, W) else cv2.resize(img, (W, H), interpolation=cv2.INTER_AREA)
def load_many(ids, path_map, workers=8):
    with ThreadPoolExecutor(workers) as ex:
        return np.stack(list(ex.map(lambda i: load_gray(path_map[i]), ids)))

t0 = time.time()
val_imgs = load_many(val_ids_eval, TRAIN_PATHS)                                              # (N, H, W) uint8
val_gt   = np.stack([np.stack([rle_to_mask(r) for r in rle_table[i]]) for i in val_ids_eval]).astype(bool)   # (N, 4, H, W)
GA       = val_gt.sum((2, 3)).astype(np.int64)                                               # true pixels per (image, defect)
n_fg, n_empty = int((GA > 0).sum()), int((GA == 0).sum())
print(f"validation images {val_imgs.shape} | GT {val_gt.shape} ({val_gt.nbytes / 1e6:.0f} MB) | {time.time() - t0:.0f}s")
print(f"positive pairs {n_fg} | empty pairs {n_empty} ({100 * n_empty / GA.size:.1f}% of {GA.size})")

# normalisation constants computed on TRAINING images (identical recipe to the project notebook -> the checkpoints expect these numbers)
sub = load_many(train_ids[: (300 if QUICK_TEST else 2000)], TRAIN_PATHS).astype(np.float32) / 255.0
PIX_MEAN, PIX_STD = float(sub.mean()), float(sub.std()); del sub
print("PIX_MEAN / PIX_STD:", round(PIX_MEAN, 4), round(PIX_STD, 4))

In [ ]:
def competition_score(inter, pa, ga):
    """OFFICIAL metric. inter / pa / ga : arrays (N images, 4 defects) = overlap, predicted and true pixel counts.
       score = 0.7*ForegroundDice + 0.2*FalsePositiveScore + 0.1*EmptyAccuracy"""
    inter, pa, ga = [np.asarray(a, dtype=np.float64) for a in (inter, pa, ga)]
    fg = ga > 0
    dice = 2 * inter / np.maximum(pa + ga, 1)
    fg_dice = dice[fg].mean() if fg.any() else 0.0
    if (~fg).any():
        fp_score  = 1 - (pa[~fg] / HW).mean()            # penalty proportional to the SIZE of the false blob
        empty_acc = (pa[~fg] == 0).mean()                # share of empty pairs where we predicted exactly nothing
    else:
        fp_score = empty_acc = 1.0
    return dict(score=0.7 * fg_dice + 0.2 * fp_score + 0.1 * empty_acc, fg_dice=fg_dice, fp_score=fp_score, empty_acc=empty_acc)

def class_gain(pa, inter, ga, n_fg_, n_empty_):
    """Contribution of ONE defect class to (score - 0.3). The official score is a pooled mean over all pairs, so
       score = 0.3 + sum_c gain_c  ->  every class can be tuned INDEPENDENTLY and the result is the exact optimum."""
    pa, inter, ga = [np.asarray(a, dtype=np.float64) for a in (pa, inter, ga)]
    fg = ga > 0
    dice_sum = (2 * inter[fg] / np.maximum(pa[fg] + ga[fg], 1)).sum()
    fp_sum   = (pa[~fg] / HW).sum()
    fa_sum   = float((pa[~fg] > 0).sum())
    return 0.7 * dice_sum / max(n_fg_, 1) - 0.2 * fp_sum / max(n_empty_, 1) - 0.1 * fa_sum / max(n_empty_, 1), dice_sum, fp_sum, fa_sum

# ---- self-checks: worked example of Milestone 1 Q8 and the all-empty baseline ----
ga_t    = np.r_[np.full(10, 1000), np.zeros(40)][:, None]
pa_t    = np.r_[np.full(10, 1000), np.full(4, 1024), np.zeros(36)][:, None]
inter_t = np.r_[np.full(6, 800), np.full(4, 400), np.zeros(40)][:, None]
m = competition_score(inter_t, pa_t, ga_t); assert abs(m["score"] - 0.7378) < 5e-4, m
z = np.zeros(GA.shape); base = competition_score(z, z, GA)["score"]
assert abs(0.3 + sum(class_gain(np.zeros(N), np.zeros(N), GA[:, c], n_fg, n_empty)[0] for c in range(NC)) - base) < 1e-9
print(f"✅ metric checks passed | predict-nothing baseline on validation = {base:.4f} (≈ 0.30 + noise)")

## 2. The post-processing engine (numpy / OpenCV, tested in isolation)
Everything after the network is cheap and **separable per class**:

1. `mask = prob > thr_c` (strict `>`; fp16 probabilities, threshold rounded to fp16 so training-time and inference-time comparisons are identical)
2. connected components (`cv2.connectedComponents`, 8- or 4-connectivity) → per component **area** and **overlap with the GT**
3. minimum-area rule — applied **per component** (small blobs removed, big ones kept) or **per pair** (the whole mask of an image×defect is dropped if its total area < A)
4. optional **confidence gate** (the pair is dropped if its most confident pixel < gate)

`component_table` runs the connected components **once** per (class, threshold); `apply_filter` then evaluates *every* min-area value from that table in milliseconds → a full threshold × min-area × gate sweep is cheap.

In [ ]:
def component_table(mask_c, gt_c, conn=8):
    """mask_c, gt_c: (N,H,W) bool for ONE class. -> flat arrays of connected components: image index, area, overlap with GT."""
    n_img = len(mask_c); imgs, areas, inters = [], [], []
    for i in np.flatnonzero(mask_c.reshape(n_img, -1).any(1)):
        n, lab = cv2.connectedComponents(mask_c[i].astype(np.uint8), connectivity=conn)      # lab: 0 = background, 1..n-1 = components
        if n <= 1: continue
        imgs.append(np.full(n - 1, i)); areas.append(np.bincount(lab.ravel(), minlength=n)[1:])
        inters.append(np.bincount(lab[gt_c[i]], minlength=n)[1:])                            # GT pixels falling in each component
    cat = lambda L: np.concatenate(L) if L else np.zeros(0, dtype=np.int64)
    return dict(img=cat(imgs).astype(np.int64), area=cat(areas).astype(np.float64), inter=cat(inters).astype(np.float64), N=n_img)

def apply_filter(tab, min_area, mode="pair"):
    """Predicted pixels and overlap per image after the minimum-area rule.  mode='component': drop components < A.
       mode='pair': drop the WHOLE mask if its total area < A."""
    N_, img, area, inter = tab["N"], tab["img"], tab["area"], tab["inter"]
    if mode == "component":
        k = area >= min_area
        return (np.bincount(img[k], weights=area[k], minlength=N_), np.bincount(img[k], weights=inter[k], minlength=N_))
    pa  = np.bincount(img, weights=area,  minlength=N_)
    it  = np.bincount(img, weights=inter, minlength=N_)
    keep = (pa >= min_area) & (pa > 0)
    return pa * keep, it * keep

def build_tables(P, GT, thr_grid=None, conn=8, strict=True):
    """P (N,4,H,W) float16 probabilities, GT (N,4,H,W) bool -> tables for every (class, threshold) + 'top' = most confident pixel per pair."""
    thr_grid = thr_grid or THR_GRID; tabs, tops = {}, {}
    for c in range(NC):
        tops[c] = P[:, c].max(axis=(1, 2)).astype(np.float32)
        for t in thr_grid:
            t16 = np.float16(t)
            m = (P[:, c] > t16) if strict else (P[:, c] >= t16)
            tabs[(c, t)] = component_table(m, GT[:, c], conn)
    return tabs, tops

def pairs_from_rules(tabs, tops, rules, n_img):
    PA, IT = np.zeros((n_img, NC)), np.zeros((n_img, NC))
    for c in range(NC):
        pa, it = apply_filter(tabs[(c, rules["thr"][c])], rules["min_area"][c], rules["mode"])
        keep = tops[c] >= np.float32(rules["gate"][c])
        PA[:, c], IT[:, c] = pa * keep, it * keep
    return PA, IT

def evaluate_rules(tabs, tops, GA_, sel, rules):
    PA, IT = pairs_from_rules(tabs, tops, rules, len(GA_))
    return competition_score(IT[sel], PA[sel], GA_[sel])

def tune(tabs, tops, GA_, sel, area_grid, gate_grid, mode="pair", thr_grid=None):
    """Exhaustive per-class search of (threshold, min-area, gate) maximising the official score on the images in `sel` (bool mask).
       Tie-break: higher gain, then fewer false alarms, then LARGER min-area (more conservative), then higher threshold."""
    thr_grid = thr_grid or THR_GRID
    g = GA_[sel]; n_fg_, n_empty_ = int((g > 0).sum()), int((g == 0).sum())
    rules = dict(thr=[], min_area=[], gate=[], mode=mode); rows = []
    for c in range(NC):
        ga, top = GA_[sel, c], tops[c][sel]; best = None
        for t in thr_grid:
            for A in area_grid:
                pa, it = apply_filter(tabs[(c, t)], A, mode); pa, it = pa[sel], it[sel]
                for gt_ in gate_grid:
                    keep = top >= np.float32(gt_)
                    gain, dsum, fps, fas = class_gain(pa * keep, it * keep, ga, n_fg_, n_empty_)
                    rows.append((CLASSES[c], t, A, gt_, gain, 0.7 * dsum / max(n_fg_, 1), 0.2 * fps / max(n_empty_, 1), 0.1 * fas / max(n_empty_, 1)))
                    key = (round(gain, 9), -fas, A, t)
                    if best is None or key > best[0]: best = (key, t, A, gt_)
        rules["thr"].append(best[1]); rules["min_area"].append(best[2]); rules["gate"].append(best[3])
    sweep = pd.DataFrame(rows, columns=["defect", "thr", "min_area", "gate", "gain", "fg_dice_part", "fp_cost", "false_alarm_cost"])
    return rules, sweep

def half_split(n_img, seed=0):
    rng = np.random.RandomState(seed); perm = rng.permutation(n_img)
    a = np.zeros(n_img, bool); a[perm[: n_img // 2]] = True
    return a, ~a

def cross_fit(tabs, tops, GA_, area_grid, gate_grid, mode):
    """HONEST estimate: tune on one half of the validation images, score on the other half (and vice-versa)."""
    a, b = half_split(len(GA_)); out = []
    for tr, te in ((a, b), (b, a)):
        r, _ = tune(tabs, tops, GA_, tr, area_grid, gate_grid, mode); out.append(evaluate_rules(tabs, tops, GA_, te, r)["score"])
    return float(np.mean(out))

# ---- fast pixel-level helpers (no connected components) used for TTA / ensembling comparisons ----
def fast_pairs(P, thr, min_area=0, strict=True, GT=None):
    """P (n,4,H,W) f16 -> predicted / overlap pixel counts (n,4) with a PAIR-level min-area rule."""
    thr = np.broadcast_to(np.asarray(thr, dtype=np.float32), (NC,)); n_img = len(P)
    PA, IT = np.zeros((n_img, NC)), np.zeros((n_img, NC))
    for c in range(NC):
        t16 = np.float16(thr[c]); m = (P[:, c] > t16) if strict else (P[:, c] >= t16)
        pa = m.sum((1, 2)).astype(np.float64); keep = (pa >= min_area) & (pa > 0)
        PA[:, c] = pa * keep
        if GT is not None: IT[:, c] = (m & GT[:, c]).sum((1, 2)) * keep
    return PA, IT
def fast_score(P, GT, GA_, thr, min_area=0, strict=True):
    PA, IT = fast_pairs(P, thr, min_area, strict, GT); return competition_score(IT, PA, GA_)
def fast_tune_thr(P, GT, GA_, grid=None, min_area=0):
    """best threshold per class by the exact per-class gain (pair-level min-area optional)."""
    grid = grid or THR_GRID; g = GA_; n_fg_, n_empty_ = int((g > 0).sum()), int((g == 0).sum()); thr = []
    for c in range(NC):
        best = None
        for t in grid:
            t16 = np.float16(t); m = P[:, c] > t16
            pa = m.sum((1, 2)).astype(np.float64); keep = (pa >= min_area) & (pa > 0); it = (m & GT[:, c]).sum((1, 2)) * keep
            gain = class_gain(pa * keep, it, g[:, c], n_fg_, n_empty_)[0]
            if best is None or round(gain, 9) > best[0]: best = (round(gain, 9), t)
        thr.append(best[1])
    return thr
print("post-processing engine ready")

## 3. Models, checkpoints and the inference core (TTA with correct un-flipping)
Architectures are **identical to the project notebook** (`SimpleUNet`, `PretrainedUNet` with ResNet34 / EfficientNet-B0/B2/B3 encoders) so that your saved `*_best.pth` files load without changes.
The inference core returns **one probability map per TTA view, already un-flipped back to the original orientation** (Question 6's bug is exactly forgetting this).

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from tqdm.auto import tqdm
device  = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_AMP = device.type == "cuda"
torch.backends.cudnn.benchmark = True
def seed_everything(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
print("device:", device, "|", torch.cuda.get_device_name(0) if USE_AMP else "NO GPU -> switch on the T4 accelerator (CPU works but is slow)")

class ConvBlock(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True),
                         nn.Conv2d(cout, cout, 3, padding=1, bias=False), nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
class SimpleUNet(nn.Module):                                             # U-Net from scratch
    def __init__(self, base=32):
        super().__init__()
        c = [base, base * 2, base * 4, base * 8, base * 16]
        self.enc = nn.ModuleList([ConvBlock(1, c[0]), ConvBlock(c[0], c[1]), ConvBlock(c[1], c[2]), ConvBlock(c[2], c[3])])
        self.bottom = ConvBlock(c[3], c[4]); self.pool = nn.MaxPool2d(2)
        self.up  = nn.ModuleList([nn.ConvTranspose2d(c[i + 1], c[i], 2, stride=2) for i in range(4)])
        self.dec = nn.ModuleList([ConvBlock(c[i] * 2, c[i]) for i in range(4)])
        self.head = nn.Conv2d(c[0], NC, 1); nn.init.constant_(self.head.bias, -4.0)
    def forward(self, x):
        skips = []
        for enc in self.enc: x = enc(x); skips.append(x); x = self.pool(x)
        x = self.bottom(x)
        for i in (3, 2, 1, 0): x = self.dec[i](torch.cat([self.up[i](x), skips[i]], 1))
        return self.head(x)                                              # raw LOGITS (B, 4, H, W)
class DecoderBlock(nn.Module):
    def __init__(self, cin, cskip, cout):
        super().__init__(); self.conv = ConvBlock(cin + cskip, cout)
    def forward(self, x, skip):
        x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([x, skip], 1))
class PretrainedUNet(nn.Module):                                         # pretrained encoder + U-Net decoder
    def __init__(self, arch, enc_ch=None, pretrained=True):
        super().__init__()
        w = "IMAGENET1K_V1" if pretrained else None
        if arch == "resnet34":
            m = torchvision.models.resnet34(weights=w); m.conv1 = self._to_gray(m.conv1)
            self.stages = nn.ModuleList([nn.Sequential(m.conv1, m.bn1, m.relu), nn.Sequential(m.maxpool, m.layer1), m.layer2, m.layer3, m.layer4])
            enc_ch = [64, 64, 128, 256, 512]
        else:
            m = getattr(torchvision.models, arch)(weights=w); m.features[0][0] = self._to_gray(m.features[0][0]); f = m.features
            self.stages = nn.ModuleList([f[0:2], f[2], f[3], f[4:6], f[6:8]])
        dec_ch, prev, self.decoder = [256, 128, 64, 32], enc_ch[-1], nn.ModuleList()
        for cskip, cout in zip(enc_ch[-2::-1], dec_ch):
            self.decoder.append(DecoderBlock(prev, cskip, cout)); prev = cout
        self.final, self.head = ConvBlock(prev, 16), nn.Conv2d(16, NC, 1); nn.init.constant_(self.head.bias, -4.0)
    @staticmethod
    def _to_gray(conv):
        new = nn.Conv2d(1, conv.out_channels, conv.kernel_size, conv.stride, conv.padding, bias=False)
        new.weight.data = conv.weight.data.sum(1, keepdim=True); return new
    def forward(self, x):
        feats = []
        for stage in self.stages: x = stage(x); feats.append(x)
        d = feats[-1]
        for block, skip in zip(self.decoder, reversed(feats[:-1])): d = block(d, skip)
        d = F.interpolate(d, scale_factor=2, mode="bilinear", align_corners=False)
        return self.head(self.final(d))
EFF = {"effb0": ("efficientnet_b0", [16, 24, 40, 112, 320]), "effb2": ("efficientnet_b2", [16, 24, 48, 120, 352]),
       "effb2b": ("efficientnet_b2", [16, 24, 48, 120, 352]), "effb3": ("efficientnet_b3", [24, 32, 48, 136, 384])}
def build_model(name, pretrained=False):
    if name == "scratch":  return SimpleUNet()
    if name == "resnet34": return PretrainedUNet("resnet34", None, pretrained)
    arch, ch = EFF[name]; return PretrainedUNet(arch, ch, pretrained)

# ---------------- inference core ----------------
VIEWS     = {"orig": (), "hflip": (3,), "vflip": (2,), "hvflip": (2, 3)}   # dims of a (B,C,H,W) tensor: 3 = WIDTH axis (left-right), 2 = HEIGHT axis (up-down)
VIEWS_ALL = ["orig", "hflip", "vflip", "hvflip"]
# NOTE: no transpose / 90-degree rotation TTA: the strip is 128 x 800, a transposed image would be 800 x 128 and look nothing like the training data.

def normalize_batch(arr_uint8):
    return torch.from_numpy((arr_uint8.astype(np.float32) / 255.0 - PIX_MEAN) / PIX_STD)[:, None]

@torch.no_grad()
def predict_views(model, x, views=VIEWS_ALL, unflip=True, as_logits=False):
    """One map per view. The flipped view is run through the net and then flipped BACK with the same dims (unflip=True)."""
    outs = []
    for v in views:
        dims = VIEWS[v]
        xin = x.flip(dims) if dims else x
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            lo = model(xin).float()
        if dims and unflip: lo = lo.flip(dims)                          # <- the line that is forgotten in the "bug"
        outs.append(lo if as_logits else torch.sigmoid(lo))
    return outs

@torch.no_grad()
def infer_val(model, views=VIEWS_ALL, bs=INFER_BATCH):
    """validation probabilities (N,4,H,W) float16: mean over the given views."""
    out = np.empty((N, NC, H, W), np.float16)
    for s in range(0, N, bs):
        x = normalize_batch(val_imgs[s:s + bs]).to(device)
        ps = predict_views(model, x, views); out[s:s + bs] = (sum(ps) / len(ps)).half().cpu().numpy()
    return out
print("models + inference core ready")

### 3.1 Find the checkpoints (or train a fallback pair)
Files named `effb0*`, `effb2*`, `effb2b*`, `effb3*`, `resnet34*`, `scratch*` (`*.pth` / `*.pt`) are searched in `/kaggle/input` and `/kaggle/working`. Duplicates (same bytes) are ignored.

In [ ]:
KEYS = ["effb2b", "effb0", "effb2", "effb3", "resnet34", "scratch"]          # 'effb2b' first: its name also starts with 'effb2'
def discover_checkpoints():
    found, seen = [], set()
    for root in (INPUT_ROOT, OUT_DIR):
        for d, _, fs in os.walk(root):
            for f in sorted(fs):
                if not f.lower().endswith((".pth", ".pt")): continue
                key = next((k for k in KEYS if f.lower().startswith(k)), None)
                if key is None: continue
                p = os.path.join(d, f); size = os.path.getsize(p)
                with open(p, "rb") as fh: head = fh.read(1 << 20)
                sig = (key, size, hashlib.md5(head).hexdigest())
                if sig in seen: continue
                seen.add(sig); found.append((key, p))
    return found
CKPTS = discover_checkpoints()
print(f"{len(CKPTS)} checkpoint(s) found:"); [print("  ", k, "->", p) for k, p in CKPTS]

# ---------- fallback: small from-scratch U-Nets so the notebook always runs end-to-end ----------
bce = nn.BCEWithLogitsLoss()
def criterion(logits, y):
    logits = logits.float(); p = torch.sigmoid(logits)
    inter, den = (p * y).sum((0, 2, 3)), p.sum((0, 2, 3)) + y.sum((0, 2, 3))
    return bce(logits, y) + (1 - (2 * inter + 1) / (den + 1)).mean()

class TrainDS(Dataset):
    def __init__(self, ids, imgs): self.ids, self.imgs = list(ids), imgs
    def __len__(self): return len(self.ids)
    def __getitem__(self, k):
        iid = self.ids[k]; img = self.imgs[iid].astype(np.float32) / 255.0
        m = np.stack([rle_to_mask(r) for r in rle_table[iid]])
        if random.random() < 0.5: img, m = img[:, ::-1], m[:, :, ::-1]
        if random.random() < 0.5: img, m = img[::-1, :], m[:, ::-1, :]
        img = (img - PIX_MEAN) / PIX_STD
        return torch.from_numpy(np.ascontiguousarray(img))[None].float(), torch.from_numpy(np.ascontiguousarray(m)).float()

def quick_train(seed, name):
    seed_everything(seed); ids = list(train_ids[:FALLBACK_IMAGES]); imgs = dict(zip(ids, load_many(ids, TRAIN_PATHS)))
    freq = has.loc[ids].mean(); w = 1 + (has.loc[ids] * (0.25 / freq)).sum(axis=1).values            # rarer defect -> drawn more often
    dl = DataLoader(TrainDS(ids, imgs), batch_size=16, sampler=WeightedRandomSampler(w, len(ids), replacement=True), num_workers=2, drop_last=True)
    model = SimpleUNet().to(device); opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    total, warm = FALLBACK_EPOCHS * len(dl), len(dl)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP); best, ck = -1, f"{OUT_DIR}/{name}_best.pth"; vi = min(N, 300)
    for ep in range(1, FALLBACK_EPOCHS + 1):
        model.train(); run = 0.0
        for x, y in tqdm(dl, desc=f"{name} epoch {ep}", leave=False):
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP): lo = model(x)
            loss = criterion(lo, y); opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step(); run += loss.item()
        model.eval(); Pv = np.empty((vi, NC, H, W), np.float16)
        with torch.no_grad():
            for s in range(0, vi, INFER_BATCH): Pv[s:s + INFER_BATCH] = predict_views(model, normalize_batch(val_imgs[s:s + INFER_BATCH]).to(device), ["orig"])[0].half().cpu().numpy()
        sc = fast_score(Pv, val_gt[:vi], GA[:vi], 0.4, 20)["score"]
        if sc > best: best = sc; torch.save(model.state_dict(), ck)
        print(f"[{name}] epoch {ep}/{FALLBACK_EPOCHS} loss {run / len(dl):.4f} | val score {sc:.4f}{'  <- best' if sc == best else ''}")
    del model, opt; torch.cuda.empty_cache()

if len(CKPTS) < 2 and FALLBACK_TRAIN:
    print(f"\nfewer than 2 checkpoints -> training {max(1, 2 - len(CKPTS))} fallback U-Net(s) from scratch (about 5 min each on a T4)")
    for sd in FALLBACK_SEEDS[: max(1, 2 - len(CKPTS))]: quick_train(sd, f"scratch_s{sd}")
    CKPTS = discover_checkpoints()
assert CKPTS, "no checkpoint available: attach the notebook(s) that produced your *_best.pth files or set FALLBACK_TRAIN = True"

### 3.2 Validation probabilities for every checkpoint, keep the best `MAX_MEMBERS`
Each candidate is run once with **4-view TTA**; only the probabilities of the best members are kept (fp16, ~1 GB each for 1,200 images). Ranking uses one fixed simple rule (thr 0.4, min-area 40) so that differences come from the model only.

In [ ]:
def load_member(key, path):
    model = build_model(key, False).to(device)
    model.load_state_dict(torch.load(path, map_location=device)); return model.eval()

MEMBERS, P4, MODELS, SIMPLE = [], {}, {}, {}
for key, path in CKPTS:
    try: model = load_member(key, path)
    except Exception as e: print("skipped", path, "->", repr(e)[:90]); continue
    name = f"{key}:{os.path.basename(os.path.dirname(path))[-18:]}/{os.path.basename(path)}"
    Pm = infer_val(model, VIEWS_ALL); sc = fast_score(Pm, val_gt, GA, 0.4, 40)["score"]; SIMPLE[name] = sc
    print(f"{name:75s} TTA-4 simple-rule score {sc:.4f}")
    MEMBERS.append(name); P4[name] = Pm; MODELS[name] = model
    if len(MEMBERS) > MAX_MEMBERS:                                       # drop the worst -> bounded RAM
        worst = min(MEMBERS, key=SIMPLE.get); MEMBERS.remove(worst); del P4[worst], MODELS[worst]; torch.cuda.empty_cache()
MEMBERS = sorted(MEMBERS, key=SIMPLE.get, reverse=True); BEST = MEMBERS[0]
print("\nensemble members (best first):", MEMBERS)
P1_BEST = infer_val(MODELS[BEST], ["orig"])                              # single best model, no TTA (for the ladder)
print("RAM used by stored probabilities: %.1f GB" % ((len(P4) + 1) * P1_BEST.nbytes / 1e9))

## 4. Threshold tuning — Questions 1 and 2
### Q1 · Logits, probability thresholds and the strict-inequality trap
* The network outputs **logits** `z`; `p = sigmoid(z)`. A probability threshold `τ` is the same as a **logit threshold** `z > ln(τ/(1-τ))` (τ = 0.5 ⇔ z > 0).
* The competition rule is **strict**: a pixel is foreground if `p > τ`. `p >= τ` differs only on exact ties — and **fp16 probabilities create many ties** (spacing ≈ 5·10⁻⁴ near 0.5), and a network that outputs exactly 0 (p = 0.5 exactly) would be flagged *everywhere* by `>=`.
* In fp16 `sigmoid(z)` **saturates at exactly 1.0** for z ≳ 8.3 → thresholds near 1 lose all ordering information in the probability domain, while logits keep it.

In [ ]:
def logit_of(p): p = float(p); return math.log(p / (1 - p))
print("τ -> logit threshold:", {t: round(logit_of(t), 4) for t in [0.1, 0.3, 0.5, 0.7, 0.9, 0.95, 0.99]})

# --- toy: a network that outputs exactly 0 logit everywhere -> p = 0.5 exactly ---
z = np.zeros((4, 8, 8), np.float32); p = (1 / (1 + np.exp(-z))).astype(np.float16)
print(f"toy zero-logit map: '>' flags {(p > np.float16(0.5)).sum()} px | '>=' flags {(p >= np.float16(0.5)).sum()} px of {p.size}")

# --- real validation probabilities of the best model (no TTA) ---
rows = []
for t in [0.3, 0.5, 0.7, 0.9, 0.99]:
    t16 = np.float16(t); strict, ge = fast_score(P1_BEST, val_gt, GA, t, 0, True), fast_score(P1_BEST, val_gt, GA, t, 0, False)
    ties = int((P1_BEST == t16).sum()); rows.append(dict(tau=t, logit_thr=round(logit_of(t), 3), tau_as_fp16=float(t16), tied_pixels=ties,
                  score_strict=round(strict["score"], 5), score_ge=round(ge["score"], 5), diff=round(ge["score"] - strict["score"], 5)))
q1 = pd.DataFrame(rows); display(q1)

# --- fp16 saturation on real logits ---
with torch.no_grad():
    zs = predict_views(MODELS[BEST], normalize_batch(val_imgs[:32]).to(device), ["orig"], as_logits=True)[0].cpu()
sat16, sat32 = float((torch.sigmoid(zs).half() == 1.0).float().mean()), float((torch.sigmoid(zs) == 1.0).float().mean())
print(f"pixels with probability EXACTLY 1.0: fp16 {100 * sat16:.4f}% | fp32 {100 * sat32:.4f}%  (max logit {zs.max():.1f})")
ANS["Q1"] = (f"logit thresholds for tau=0.3/0.5/0.7/0.9: " + ", ".join(f"{logit_of(t):.4f}" for t in [0.3, 0.5, 0.7, 0.9]) +
             f"; ties at tau=0.5: {int(q1.loc[q1.tau == 0.5, 'tied_pixels'].iloc[0])}; score strict {q1.loc[q1.tau == 0.5, 'score_strict'].iloc[0]} vs >= {q1.loc[q1.tau == 0.5, 'score_ge'].iloc[0]}; "
             f"zero-logit toy: > flags {(p > np.float16(0.5)).sum()}, >= flags {(p >= np.float16(0.5)).sum()}")
print("ANSWER Q1:", ANS["Q1"])

### Q2 · Per-class threshold / min-area selection from a sweep table
We sweep **threshold × min-area × gate** for every class on the best single model (no TTA) and select per class the row with the highest **gain** (= that class's exact contribution to the official score). Because `score = 0.3 + Σ_c gain_c`, picking the best row of each class independently gives the best possible combined score.
The table also shows *why* the optimum differs by class: rare / small defects prefer low thresholds (recall-oriented, a missed defect costs ≈ 0.7/n_fg while a small false blob costs ≈ 0.1/n_empty).

In [ ]:
t0 = time.time()
TAB_O, TOP_O = build_tables(P1_BEST, val_gt, THR_GRID, conn=8)
print(f"connected-component tables built in {time.time() - t0:.0f}s")
SEL_ALL = np.ones(N, bool)
rules_q2, sweep_q2 = tune(TAB_O, TOP_O, GA, SEL_ALL, AREA_GRID, [0.0], "pair")          # gate off here: pure threshold x min-area table
sel_rows = []
for c, cls in enumerate(CLASSES):
    s = sweep_q2[sweep_q2.defect == cls]
    b = s[(s.thr == rules_q2["thr"][c]) & (s.min_area == rules_q2["min_area"][c])].iloc[0]
    d_default = s[(s.thr == 0.5) & (s.min_area == 0)].iloc[0]
    sel_rows.append(dict(defect=cls, best_thr=rules_q2["thr"][c], best_min_area=rules_q2["min_area"][c], gain_best=round(b.gain, 5), gain_at_0p5_no_area=round(d_default.gain, 5),
                         improvement=round(b.gain - d_default.gain, 5), fg_dice_part=round(b.fg_dice_part, 5), fp_cost=round(b.fp_cost, 6), false_alarm_cost=round(b.false_alarm_cost, 6)))
q2 = pd.DataFrame(sel_rows).set_index("defect"); display(q2)
for cls in CLASSES[:4]:
    pv = sweep_q2[sweep_q2.defect == cls].pivot(index="thr", columns="min_area", values="gain").round(4)
    print(f"\n{cls}: gain table (rows = threshold, columns = min-area px)  -> best cell is the selection"); display(pv)
r_q2 = evaluate_rules(TAB_O, TOP_O, GA, SEL_ALL, rules_q2); r_def = evaluate_rules(TAB_O, TOP_O, GA, SEL_ALL, dict(thr=[0.5] * 4, min_area=[0] * 4, gate=[0.0] * 4, mode="pair"))
print(f"official score: global thr 0.5, no filter = {r_def['score']:.4f}  ->  per-class tuned = {r_q2['score']:.4f}   (check: 0.3 + sum of best gains = {0.3 + q2.gain_best.sum():.4f})")
ANS["Q2"] = "; ".join(f"{cls}: thr {rules_q2['thr'][c]}, min_area {rules_q2['min_area'][c]}" for c, cls in enumerate(CLASSES)) + f"; score {r_def['score']:.4f} -> {r_q2['score']:.4f}"
print("ANSWER Q2:", ANS["Q2"])

## 5. Connected components, connectivity and area filtering — Question 3
* **4-connectivity**: pixels touch only through an edge; **8-connectivity**: edges *and* diagonals. A diagonal chain of pixels is **1** component with 8-connectivity but **many** with 4-connectivity → with a min-area rule, 4-connectivity throws away thin diagonal defects.
* `cv2.connectedComponents(mask_uint8, connectivity=8)` returns `(n_labels, labels)` where label 0 is the **background**, so there are `n_labels - 1` components.
* Filtering rule used here: `keep component if area >= A` (a component of exactly A pixels is **kept**).

In [ ]:
# --- toy: diagonal chain ---
toy = np.eye(5, dtype=np.uint8)
n8, _ = cv2.connectedComponents(toy, connectivity=8); n4, _ = cv2.connectedComponents(toy, connectivity=4)
print(f"toy 5x5 diagonal: components 8-conn = {n8 - 1}, 4-conn = {n4 - 1}  (label 0 = background is NOT counted)")

# --- ground-truth masks of the WHOLE training set: components and areas per class, 4 vs 8 connectivity ---
GT_COMP = {c: {4: [], 8: []} for c in CLASSES}; PAIR_AREA = {c: [] for c in CLASSES}
t0 = time.time()
for iid in all_ids:
    for c, cls in enumerate(CLASSES):
        r = rle_table[iid][c]
        if not r.strip(): continue
        m = rle_to_mask(r); PAIR_AREA[cls].append(int(m.sum()))
        for conn in (4, 8):
            n, lab = cv2.connectedComponents(m, connectivity=conn); GT_COMP[cls][conn] += list(np.bincount(lab.ravel(), minlength=n)[1:])
print(f"GT components computed in {time.time() - t0:.0f}s")
rows = []
for cls in CLASSES:
    a8, a4 = np.array(GT_COMP[cls][8]), np.array(GT_COMP[cls][4])
    rows.append(dict(defect=cls, positive_pairs=len(PAIR_AREA[cls]), comps_8conn=len(a8), comps_4conn=len(a4), min_area_8conn=int(a8.min()), median_area_8conn=int(np.median(a8)),
                     **{f"comps<{A}px (8c)": int((a8 < A).sum()) for A in (20, 50, 100)}, **{f"px lost % A={A}": round(100 * a8[a8 < A].sum() / a8.sum(), 2) for A in (20, 50, 100)}))
q3 = pd.DataFrame(rows).set_index("defect"); display(q3.T)

# --- sanity: table-based filter == explicit cv2 stats-based removal (on real predictions) ---
rng = np.random.RandomState(1)
for _ in range(30):
    c, i, A, t = rng.randint(NC), rng.randint(N), [20, 50, 100][rng.randint(3)], 0.3
    m = (P1_BEST[i, c] > np.float16(t)).astype(np.uint8); n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    ref = np.isin(lab, [k for k in range(1, n) if st[k, cv2.CC_STAT_AREA] >= A]); pa, it = apply_filter(TAB_O[(c, t)], A, "component")
    assert pa[i] == ref.sum() and it[i] == (ref & val_gt[i, c]).sum()
print("✅ apply_filter() == cv2.connectedComponentsWithStats removal")
ANS["Q3"] = (f"toy diagonal: 8-conn {n8 - 1} component(s), 4-conn {n4 - 1}; GT components (8-conn) per class: " + ", ".join(f"{c}={q3.loc[c, 'comps_8conn']}" for c in CLASSES) +
             "; (4-conn): " + ", ".join(f"{c}={q3.loc[c, 'comps_4conn']}" for c in CLASSES) +
             "; components < 50 px (8c): " + ", ".join(f"{c}={q3.loc[c, 'comps<50px (8c)']}" for c in CLASSES))
print("ANSWER Q3:", ANS["Q3"])

## 6. Does post-processing actually improve the official score? — Question 4 (ablation ladder)
**One change per row**, everything else fixed, same validation images, official metric. Each tuned row is tuned *on the validation images* (so it is slightly optimistic); the **cross-fit** column tunes on one half and scores the other half → the honest number.
Rows: raw model → per-class threshold → + pair-level min-area → component-level min-area instead → + confidence gate → + TTA (4 views) → + ensemble (soft average).
Watch the three components, not just the total: a filter may *raise* FP-score / Empty-accuracy and *lower* foreground Dice.

In [ ]:
def combine(Ps, method="soft", chunk=100):
    """average several (N,4,H,W) float16 probability arrays: 'soft' = mean of probabilities, 'logit' = mean of logits (sigmoid back)."""
    out = np.empty_like(Ps[0])
    for s in range(0, len(out), chunk):
        blk = [p[s:s + chunk].astype(np.float32) for p in Ps]
        if method == "soft": r = sum(blk) / len(blk)
        else:
            L = [np.log(np.clip(b, 1e-4, 1 - 1e-4) / (1 - np.clip(b, 1e-4, 1 - 1e-4))) for b in blk]; r = 1 / (1 + np.exp(-sum(L) / len(L)))
        out[s:s + chunk] = r.astype(np.float16)
    return out

LADDER = []
def add_row(name, tabs, tops, rules, honest=None):
    m = evaluate_rules(tabs, tops, GA, SEL_ALL, rules)
    LADDER.append(dict(step=name, score=m["score"], fg_dice=m["fg_dice"], fp_score=m["fp_score"], empty_acc=m["empty_acc"], cross_fit_score=honest, rules=rules))

R_RAW = dict(thr=[0.5] * 4, min_area=[0] * 4, gate=[0.0] * 4, mode="pair")
add_row("L0  raw best model, p > 0.5, no post-processing", TAB_O, TOP_O, R_RAW)
r1, _ = tune(TAB_O, TOP_O, GA, SEL_ALL, [0], [0.0], "pair");              add_row("L1  + per-class threshold", TAB_O, TOP_O, r1, cross_fit(TAB_O, TOP_O, GA, [0], [0.0], "pair"))
r2, _ = tune(TAB_O, TOP_O, GA, SEL_ALL, AREA_GRID, [0.0], "pair");        add_row("L2  + minimum area (whole pair dropped if total area < A)", TAB_O, TOP_O, r2, cross_fit(TAB_O, TOP_O, GA, AREA_GRID, [0.0], "pair"))
r3, _ = tune(TAB_O, TOP_O, GA, SEL_ALL, AREA_GRID, [0.0], "component");   add_row("L3  min-area per connected COMPONENT instead of per pair", TAB_O, TOP_O, r3, cross_fit(TAB_O, TOP_O, GA, AREA_GRID, [0.0], "component"))
BEST_MODE = "component" if LADDER[3]["score"] > LADDER[2]["score"] else "pair"; print("min-area mode kept for the next rows:", BEST_MODE)
r4, _ = tune(TAB_O, TOP_O, GA, SEL_ALL, AREA_GRID, GATE_GRID, BEST_MODE); add_row("L4  + confidence gate", TAB_O, TOP_O, r4, cross_fit(TAB_O, TOP_O, GA, AREA_GRID, GATE_GRID, BEST_MODE))

t0 = time.time(); TAB_T, TOP_T = build_tables(P4[BEST], val_gt, THR_GRID, 8)                      # best single model + 4-view TTA
r5, _ = tune(TAB_T, TOP_T, GA, SEL_ALL, AREA_GRID, GATE_GRID, BEST_MODE); add_row("L5  + TTA (orig, hflip, vflip, hvflip; un-flipped)", TAB_T, TOP_T, r5, cross_fit(TAB_T, TOP_T, GA, AREA_GRID, GATE_GRID, BEST_MODE))
if len(MEMBERS) > 1:
    P_ENS = combine([P4[m] for m in MEMBERS], "soft"); TAB_E, TOP_E = build_tables(P_ENS, val_gt, THR_GRID, 8)
    r6, _ = tune(TAB_E, TOP_E, GA, SEL_ALL, AREA_GRID, GATE_GRID, BEST_MODE); add_row(f"L6  + ensemble of {len(MEMBERS)} models (soft average of probabilities)", TAB_E, TOP_E, r6, cross_fit(TAB_E, TOP_E, GA, AREA_GRID, GATE_GRID, BEST_MODE))
else:
    P_ENS, TAB_E, TOP_E, r6 = P4[BEST], TAB_T, TOP_T, r5; print("only one member available -> L6 not run (no ensemble possible)")
print(f"L5 / L6 tables built in {time.time() - t0:.0f}s")

lad = pd.DataFrame(LADDER).drop(columns="rules").set_index("step"); lad["cross_fit_score"] = pd.to_numeric(lad["cross_fit_score"], errors="coerce")
lad.insert(0, "delta_vs_prev", lad.score.diff()); lad.insert(1, "delta_cross_fit", lad.cross_fit_score.diff()); display(lad.round(4))
lad.round(5).to_csv(f"{OUT_DIR}/ablation_ladder_m5.csv")
plt.figure(figsize=(10, 3.6)); plt.barh(list(lad.index)[::-1], list(lad.score)[::-1], color="#4c78a8"); plt.xlim(lad.score.min() - 0.03, lad.score.max() + 0.01)
plt.xlabel("official validation score (tuned on the same images)"); plt.title("Post-processing / TTA / ensemble ladder"); plt.tight_layout(); plt.show()

# --- 4- vs 8-connectivity on the final-candidate probabilities (component mode, same tuning) ---
TAB_E4, TOP_E4 = build_tables(P_ENS, val_gt, THR_GRID, 4)
r4c, _ = tune(TAB_E4, TOP_E4, GA, SEL_ALL, AREA_GRID, GATE_GRID, BEST_MODE); s4 = evaluate_rules(TAB_E4, TOP_E4, GA, SEL_ALL, r4c)["score"]; s8 = LADDER[-1]["score"]
print(f"connectivity on the final candidate: 8-conn {s8:.4f} | 4-conn {s4:.4f}  (only matters in component mode)")
ANS["Q4"] = "; ".join(f"{r['step'].split('  ')[0]} {r['score']:.4f} (FG {r['fg_dice']:.4f}, FP {r['fp_score']:.4f}, EA {r['empty_acc']:.4f})" for r in LADDER) + f"; min-area mode kept: {BEST_MODE}; conn8 {s8:.4f} vs conn4 {s4:.4f}"
print("ANSWER Q4:", ANS["Q4"])

## 7. Recall ceiling of min-area filtering at the pair level — Question 5
Even a **perfect** model loses every pair whose true mask has fewer than `A` pixels if the filter is applied at the pair level (the mask is predicted, then deleted). So the foreground Dice can never exceed the share of positive pairs with `area >= A`, and the score can never exceed `0.7 × kept_share + 0.3`.
This is the price side of the trade-off whose benefit is the FP-score / Empty-accuracy gain measured in Question 4.

In [ ]:
rows = []
allpairs = np.concatenate([np.array(PAIR_AREA[c]) for c in CLASSES])
for A in [10, 20, 40, 50, 100, 200, 300, 500]:
    row = dict(min_area=A)
    for c in CLASSES: a = np.array(PAIR_AREA[c]); row[f"{c} pairs lost %"] = round(100 * (a < A).mean(), 2)
    kept = (allpairs >= A).mean()
    row.update({"ALL pairs lost %": round(100 * (1 - kept), 2), "max FG-Dice (perfect model)": round(kept, 4), "max official score": round(0.3 + 0.7 * kept, 4),
                "GT pixels in comps < A (8c) %": round(100 * sum(np.array(GT_COMP[c][8])[np.array(GT_COMP[c][8]) < A].sum() for c in CLASSES) / sum(sum(GT_COMP[c][8]) for c in CLASSES), 2)})
    rows.append(row)
q5 = pd.DataFrame(rows).set_index("min_area"); display(q5)
print("smallest true pair area per class:", {c: int(min(PAIR_AREA[c])) for c in CLASSES})
v = {A: round(100 * float((GA[GA > 0] < A).mean()), 2) for A in (20, 50, 100)}; print("validation positive pairs lost (pair level) for A=20/50/100 (%):", v)
ANS["Q5"] = "; ".join(f"A={A}: {q5.loc[A, 'ALL pairs lost %']}% of positive pairs lost, ceiling FG-Dice {q5.loc[A, 'max FG-Dice (perfect model)']}, ceiling score {q5.loc[A, 'max official score']}" for A in (20, 50, 100)) + "; per class lost % at A=50: " + ", ".join(f"{c}={q5.loc[50, f'{c} pairs lost %']}" for c in CLASSES)
print("ANSWER Q5:", ANS["Q5"])

## 8. Test-time augmentation — Question 6
A flipped input gives a flipped output, so **every flipped prediction must be flipped back with the same axis** before averaging. On a `(B, C, H, W)` tensor: **dim 3 = width = left-right flip**, **dim 2 = height = up-down flip**, `flip(2, 3)` = both (a 180° rotation).
The bug: averaging the flipped prediction **without** flipping it back (or flipping back along the wrong axis) blends a mirrored map with the right one → blurred / misplaced masks and a worse score than not using TTA at all.
All variants below use the **same fixed rule** (thr 0.4, min-area 40) on the same images, so only the TTA changes.

In [ ]:
VS = min(N, 300); model = MODELS[BEST]
ok = {v: np.empty((VS, NC, H, W), np.float16) for v in VIEWS_ALL}; bug = {v: np.empty((VS, NC, H, W), np.float16) for v in VIEWS_ALL}; wrong_h = np.empty((VS, NC, H, W), np.float16)
for s in range(0, VS, INFER_BATCH):
    x = normalize_batch(val_imgs[s:min(s + INFER_BATCH, VS)]).to(device); e = s + len(x)
    for v, p in zip(VIEWS_ALL, predict_views(model, x, VIEWS_ALL, unflip=True)):  ok[v][s:e]  = p.half().cpu().numpy()
    for v, p in zip(VIEWS_ALL, predict_views(model, x, VIEWS_ALL, unflip=False)): bug[v][s:e] = p.half().cpu().numpy()
    with torch.no_grad(): wrong_h[s:e] = predict_views(model, x, ["hflip"], unflip=False)[0].flip(2).half().cpu().numpy()     # un-flipped along the WRONG axis
mean = lambda L: (sum(a.astype(np.float32) for a in L) / len(L)).astype(np.float16)
V = {"orig (no TTA)": ok["orig"], "hflip view alone, un-flipped": ok["hflip"], "vflip view alone, un-flipped": ok["vflip"], "hv view alone, un-flipped": ok["hvflip"],
     "TTA 2: orig + hflip": mean([ok["orig"], ok["hflip"]]), "TTA 2: orig + vflip": mean([ok["orig"], ok["vflip"]]), "TTA 4 (correct)": mean([ok[v] for v in VIEWS_ALL]),
     "BUG: orig + hflip, hflip NOT un-flipped": mean([ok["orig"], bug["hflip"]]), "BUG: orig + hflip, un-flipped on WRONG axis (dim 2)": mean([ok["orig"], wrong_h]),
     "BUG: TTA 4, none un-flipped": mean([bug[v] for v in VIEWS_ALL])}
rows = []
for k, Pv in V.items():
    m = fast_score(Pv, val_gt[:VS], GA[:VS], 0.4, 40)
    rows.append(dict(variant=k, score=m["score"], fg_dice=m["fg_dice"], fp_score=m["fp_score"], empty_acc=m["empty_acc"],
                     mean_abs_diff_vs_orig=float(np.abs(Pv.astype(np.float32) - ok["orig"].astype(np.float32)).mean())))
q6 = pd.DataFrame(rows).set_index("variant"); q6["delta_vs_orig"] = q6.score - q6.loc["orig (no TTA)", "score"]; display(q6.round(5))
print("equivariance check: mean |p(orig) - unflip(p(flip))| =", {v: round(float(np.abs(ok[v].astype(np.float32) - ok['orig'].astype(np.float32)).mean()), 5) for v in VIEWS_ALL[1:]},
      "  (small = the network is nearly flip-equivariant; the bug is much larger)")
fig, ax = plt.subplots(figsize=(10, 3.8)); ax.barh(list(q6.index)[::-1], list(q6.score)[::-1], color=["#e45756" if "BUG" in k else "#4c78a8" for k in list(q6.index)[::-1]])
ax.set_xlim(q6.score.min() - 0.02, q6.score.max() + 0.01); ax.set_title("TTA variants (red = the 'forgot to un-flip' bug)"); plt.tight_layout(); plt.show()
ANS["Q6"] = (f"axes: dim 3 (width) = hflip, dim 2 (height) = vflip, flip(2,3) = both; orig {q6.loc['orig (no TTA)', 'score']:.4f}; TTA2 hflip {q6.loc['TTA 2: orig + hflip', 'score']:.4f}; TTA2 vflip {q6.loc['TTA 2: orig + vflip', 'score']:.4f}; "
             f"TTA4 {q6.loc['TTA 4 (correct)', 'score']:.4f}; BUG hflip not un-flipped {q6.loc['BUG: orig + hflip, hflip NOT un-flipped', 'score']:.4f}; BUG wrong axis {q6.loc['BUG: orig + hflip, un-flipped on WRONG axis (dim 2)', 'score']:.4f}; BUG none un-flipped {q6.loc['BUG: TTA 4, none un-flipped', 'score']:.4f}")
print("ANSWER Q6:", ANS["Q6"]); del ok, bug, wrong_h, V

## 9. Ensembling: soft averaging, hard voting and where to average — Question 7
* **Soft averaging** = mean of the models' **probabilities**, *then* threshold (keeps each model's confidence).
* **Logit averaging** = mean of logits, then sigmoid (a geometric-style mean; one very confident model counts more).
* **Hard voting** = threshold each model **first**, then keep a pixel if ≥ *k* models agree (k = 1 *union*, majority, k = M *intersection*). Confidence is lost → coarser, but robust to badly calibrated members.
* **Where to average**: after the un-flip (TTA views of the *same* model), before the threshold, in probability (or logit) space, never on already-thresholded masks if you want to tune a threshold afterwards.
Each method gets its own best per-class threshold (pixel level, no min-area) so the comparison is fair.

In [ ]:
def tune_votes(Ps, ks, grid=None):
    """per-class threshold search for hard voting: each member is thresholded at t, a pixel is kept if >= k members agree."""
    grid = grid or THR_GRID; res = {k: dict(thr=[], gain=0.0) for k in ks}
    for c in range(NC):
        best = {k: None for k in ks}
        for t in grid:
            t16 = np.float16(t); votes = sum((P[:, c] > t16).astype(np.uint8) for P in Ps)
            for k in ks:
                m = votes >= k; pa = m.sum((1, 2)).astype(np.float64); it = (m & val_gt[:, c]).sum((1, 2)).astype(np.float64)
                gain = class_gain(pa, it, GA[:, c], n_fg, n_empty)[0]
                if best[k] is None or round(gain, 9) > best[k][0]: best[k] = (round(gain, 9), t)
        for k in ks: res[k]["thr"].append(best[k][1]); res[k]["gain"] += best[k][0]
    return {k: (res[k]["thr"], 0.3 + res[k]["gain"]) for k in ks}

def soft_score(P_):                                                       # best thr per class, then official score (pixel level, no min-area)
    thr = fast_tune_thr(P_, val_gt, GA); return thr, fast_score(P_, val_gt, GA, thr)["score"]
rows = []
for m in MEMBERS: thr, sc = soft_score(P4[m]); rows.append(dict(method=f"single (TTA-4): {m}", thresholds=thr, score=sc))
if len(MEMBERS) >= 2:
    M = len(MEMBERS); thr, sc = soft_score(P_ENS); rows.append(dict(method=f"SOFT average of {M} probability maps", thresholds=thr, score=sc))
    P_LOG = combine([P4[m] for m in MEMBERS], "logit"); thr, sc = soft_score(P_LOG); rows.append(dict(method=f"LOGIT average of {M} models", thresholds=thr, score=sc)); del P_LOG
    ks = sorted({1, M // 2 + 1, M}); vt = tune_votes([P4[m] for m in MEMBERS], ks)
    for k in ks: rows.append(dict(method=f"HARD vote, k >= {k} of {M} ({'union' if k == 1 else 'intersection' if k == M else 'majority'})", thresholds=vt[k][0], score=vt[k][1]))
    # diversity: how much do the members' masks agree? (pairwise Dice of binary predictions at thr 0.4)
    dv = []
    for a, b in itertools.combinations(MEMBERS, 2):
        A_, B_ = P4[a][:200] > np.float16(0.4), P4[b][:200] > np.float16(0.4); dv.append(2 * (A_ & B_).sum() / max((A_.sum() + B_.sum()), 1))
    print(f"mean pairwise Dice between members' masks (thr 0.4): {np.mean(dv):.3f}  (lower = more diverse = more to gain from averaging)")
q7 = pd.DataFrame(rows).set_index("method"); q7["delta_vs_best_single"] = q7.score - q7.score[[r.startswith("single") for r in q7.index]].max(); display(q7.round(5))
ANS["Q7"] = "; ".join(f"{k}: {r.score:.4f}" for k, r in q7.iterrows())
print("ANSWER Q7:", ANS["Q7"])
if len(MEMBERS) < 2: print("(only one member found: attach more checkpoints to compare ensembling methods)")

## 10. The inference pipeline (streamed) and the Kaggle budget — Question 8
**Order of operations, from raw image to RLE** (any other order is a bug):
`read image → normalise (train mean/std) → for each model, for each TTA view: forward → un-flip → sigmoid → average (views × models) → threshold (strict >) → connected-components min-area filter → gate → RLE encode (column-major, 1-based) → row in sample_submission order`.

**Why streaming:** holding all test probabilities would need `N_test × 4 × 128 × 800 × bytes` per model-view (≈ 2.5 GB in fp16 for 3,000 images, **per model and per view**). Batches are processed and discarded instead, so memory is independent of the number of test images; time grows linearly with `models × views`.

In [ ]:
def postprocess_np(prob16, rules, conn=8):
    """prob16 (B,4,H,W) float16 numpy -> uint8 masks. Mirrors the sweep engine exactly: strict '>' on fp16, component / pair min-area, gate."""
    out = np.zeros(prob16.shape, np.uint8)
    for c in range(NC):
        t16, A, g = np.float16(rules["thr"][c]), rules["min_area"][c], np.float32(rules["gate"][c])
        m = prob16[:, c] > t16
        for b in np.flatnonzero(m.reshape(len(m), -1).any(1)):
            if prob16[b, c].max().astype(np.float32) < g: continue
            mb = m[b].astype(np.uint8)
            if rules["mode"] == "component":
                n, lab, st, _ = cv2.connectedComponentsWithStats(mb, connectivity=conn)
                keep = [k for k in range(1, n) if st[k, cv2.CC_STAT_AREA] >= A]
                if keep: out[b, c] = np.isin(lab, keep)
            elif mb.sum() >= A: out[b, c] = mb
    return out

@torch.no_grad()
def batch_probs(models, x, views):
    acc, n = None, 0
    for m in models:
        for p in predict_views(m, x, views): acc = p if acc is None else acc + p; n += 1
    return (acc / n).half().cpu().numpy()                                       # fp16, exactly like the stored validation probabilities

def run_pipeline(get_batch, n_items, models, views, rules, bs=INFER_BATCH, conn=8, desc="predicting"):
    rles = []
    for s in tqdm(range(0, n_items, bs), desc=desc, leave=False):
        arr = get_batch(s, min(s + bs, n_items)); x = normalize_batch(arr).to(device)
        pred = postprocess_np(batch_probs(models, x, views), rules, conn)
        rles.extend([[mask_to_rle(pred[b, c]) for c in range(NC)] for b in range(len(arr))])
    return rles

N_TEST = sample_df.image_id.nunique(); print("test images:", N_TEST, "| rows:", len(sample_df))

In [ ]:
def bench(models, views, bs=INFER_BATCH, reps=3):
    """measured GPU time per image and peak GPU memory of one forward configuration (models x views)."""
    x = normalize_batch(val_imgs[:bs]).to(device)
    batch_probs(models, x, views)                                                # warm-up (cudnn autotune)
    if USE_AMP: torch.cuda.synchronize(); base = torch.cuda.memory_allocated() / 2**20; torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    for _ in range(reps): batch_probs(models, x, views)
    if USE_AMP: torch.cuda.synchronize()
    ms = 1000 * (time.time() - t0) / reps / len(x)
    peak = torch.cuda.max_memory_allocated() / 2**20 if USE_AMP else float("nan")
    return ms, peak, (peak - base if USE_AMP else float("nan"))

# --- measured per-image costs of the CPU / IO parts ---
xb = P4[BEST][:INFER_BATCH]; t0 = time.time(); pm = postprocess_np(xb, r6 if len(MEMBERS) > 1 else r5); _ = [mask_to_rle(pm[b, c]) for b in range(len(xb)) for c in range(NC)]
POST_MS = 1000 * (time.time() - t0) / len(xb)
test_ids_all = list(dict.fromkeys(sample_df.image_id)); t0 = time.time(); _ = load_many(test_ids_all[:INFER_BATCH], TEST_PATHS); IO_MS = 1000 * (time.time() - t0) / INFER_BATCH
print(f"CPU post-processing + RLE: {POST_MS:.2f} ms/img | image reading: {IO_MS:.2f} ms/img")

M_ = len(MEMBERS); CONFIGS = {f"A: best model, no TTA": ([BEST], ["orig"]), f"B: best model, TTA-4": ([BEST], VIEWS_ALL),
                              f"C: {M_} models, no TTA": (MEMBERS, ["orig"]), f"D: {M_} models, TTA-4": (MEMBERS, VIEWS_ALL)}
rows = []
for name, (ms, views) in CONFIGS.items():
    models = [MODELS[m] for m in ms]; gpu_ms, peak, act = bench(models, views)
    params = sum(sum(p.numel() for p in m.parameters()) for m in models) / 1e6
    total_ms = gpu_ms + POST_MS + IO_MS                                            # reading / post-processing are sequential here (worst case)
    store_gb = N_TEST * NC * HW * 2 * len(ms) * len(views) / 1e9                   # if ALL fp16 probability maps were kept in RAM (the thing we do NOT do)
    rows.append({"config": name, "forward passes / image": len(ms) * len(views), "params (M)": round(params, 1), "weights fp32 (MB)": round(params * 4),
                 "GPU ms / image": round(gpu_ms, 1), "GPU peak (MB)": round(peak), "activation peak (MB)": round(act), "est. test time (min)": round(N_TEST * total_ms / 60000, 1),
                 "naive store-all RAM (GB)": round(store_gb, 1), "streamed RAM (MB / batch)": round(INFER_BATCH * NC * HW * 4 * 2 / 2**20),
                 "fits time": N_TEST * total_ms / 60000 <= TIME_LIMIT_MIN, "fits GPU": (peak <= GPU_MEM_LIMIT_MB) if USE_AMP else None, "naive RAM fits": store_gb <= RAM_LIMIT_GB})
q8 = pd.DataFrame(rows).set_index("config"); display(q8.T)
print(f"formula: time ≈ N_test × (forward_passes × t_forward + t_post + t_io) | naive RAM = N_test × 4 × H × W × bytes × models × views | limits used: {TIME_LIMIT_MIN} min, {GPU_MEM_LIMIT_MB} MB GPU, {RAM_LIMIT_GB} GB RAM")
ANS["Q8"] = "; ".join(f"{k}: {r['forward passes / image']} passes/img, GPU peak {r['GPU peak (MB)']} MB, est. {r['est. test time (min)']} min, naive RAM {r['naive store-all RAM (GB)']} GB, fits time {r['fits time']}" for k, r in q8.iterrows())
print("ANSWER Q8:", ANS["Q8"])

## 11. Final configuration, end-to-end proof on validation, test inference — Question 9
The final configuration is the candidate with the best **cross-fit (honest)** validation score among *best model + TTA-4* and *ensemble + TTA-4*. Before touching the test set the **whole streamed pipeline** (images → models → un-flip → average → threshold → min-area → RLE → decode) is run on validation images and must reproduce the sweep-engine score — this proves the order of operations and the RLE are right.

In [ ]:
cand = {"best model + TTA-4": (r5, [BEST], LADDER[5]["cross_fit_score"], LADDER[5]["score"])}      # row L5 = best single model + TTA-4
if len(MEMBERS) > 1: cand[f"ensemble of {len(MEMBERS)} + TTA-4"] = (r6, MEMBERS, LADDER[-1]["cross_fit_score"], LADDER[-1]["score"])
for k, v in cand.items(): print(f"{k:32s} tuned score {v[3]:.4f} | cross-fit (honest) {v[2]:.4f}")
FINAL_NAME = max(cand, key=lambda k: cand[k][2]); FINAL_RULES, FINAL_MEMBERS = cand[FINAL_NAME][0], cand[FINAL_NAME][1]; FINAL_VIEWS = VIEWS_ALL
print("\nFINAL:", FINAL_NAME, "\n  members:", FINAL_MEMBERS, "\n  views  :", FINAL_VIEWS, "\n  rules  :", FINAL_RULES)

# ---- end-to-end proof on validation images (RLE -> decode -> official score) ----
NRT = min(N, 300); t0 = time.time()
rles_val = run_pipeline(lambda s, e: val_imgs[s:e], NRT, [MODELS[m] for m in FINAL_MEMBERS], FINAL_VIEWS, FINAL_RULES, desc="validation round-trip")
PA_rt = np.array([[int(rle_to_mask(r).sum()) for r in row] for row in rles_val], float)
IT_rt = np.array([[int((rle_to_mask(r).astype(bool) & val_gt[i, c]).sum()) for c, r in enumerate(row)] for i, row in enumerate(rles_val)], float)
rt = competition_score(IT_rt, PA_rt, GA[:NRT])
tabs_f, tops_f = (TAB_E, TOP_E) if FINAL_NAME.startswith("ensemble") else (TAB_T, TOP_T)
sel_rt = np.zeros(N, bool); sel_rt[:NRT] = True; eng = evaluate_rules(tabs_f, tops_f, GA, sel_rt, FINAL_RULES)
print(f"streamed pipeline on {NRT} val images: score {rt['score']:.4f} | sweep engine on the same images: {eng['score']:.4f} | |diff| = {abs(rt['score'] - eng['score']):.5f} | {time.time() - t0:.0f}s")
assert abs(rt["score"] - eng["score"]) < 2e-3, "pipeline and sweep engine disagree -> order of operations / un-flip / RLE bug"
print("✅ end-to-end pipeline reproduces the tuned validation score")

In [ ]:
# ---------------- predict the test set (streamed) ----------------
t0 = time.time()
rles_test = run_pipeline(lambda s, e: load_many(test_ids_all[s:e], TEST_PATHS), len(test_ids_all), [MODELS[m] for m in FINAL_MEMBERS], FINAL_VIEWS, FINAL_RULES, desc="test images")
TEST_MIN = (time.time() - t0) / 60; print(f"test inference: {TEST_MIN:.1f} min for {len(test_ids_all)} images ({60 * TEST_MIN / len(test_ids_all) * 1000:.0f} ms/img)")
rle_of = {(iid, cls): rles_test[i][c] for i, iid in enumerate(test_ids_all) for c, cls in enumerate(CLASSES)}
sub = sample_df.copy(); sub["mask_rle"] = [(rle_of[(i, c)] or " ") for i, c in zip(sub.image_id, sub.defect_type)]        # empty mask = single space, like the sample
sub[["ID", "image_id", "defect_type", "mask_rle"]].to_csv(f"{OUT_DIR}/submission.csv", index=False)

# ---------------- submission integrity: every rule of the competition contract ----------------
def validate_submission(path, sample):
    s = pd.read_csv(path, keep_default_na=False); checks = []
    def chk(name, ok, detail=""): checks.append((name, bool(ok), detail))
    chk("columns == sample_submission columns (same order)", list(s.columns) == list(sample.columns), str(list(s.columns)))
    chk("row count == sample", len(s) == len(sample), f"{len(s)} vs {len(sample)}")
    chk("ID column identical and in the same order", (s.ID.values == sample.ID.values).all())
    chk("image_id / defect_type identical row by row", (s.image_id.values == sample.image_id.values).all() and (s.defect_type.values == sample.defect_type.values).all())
    chk("one row per (image_id, defect_type): no duplicates", not s.duplicated(["image_id", "defect_type"]).any())
    chk("every image has exactly the 4 defect types", (s.groupby("image_id").defect_type.nunique() == NC).all())
    chk("no NaN / missing mask_rle", s.mask_rle.notna().all())
    bad_fmt = bad_run = bad_rt = 0; n_empty_ = 0
    for r in s.mask_rle:
        if r.strip() == "": n_empty_ += 1; bad_fmt += (r != " "); continue
        try: v = np.array(r.split(), dtype=np.int64)
        except ValueError: bad_fmt += 1; continue
        st, ln = v[0::2], v[1::2]
        if len(v) % 2 or (st < 1).any() or (ln < 1).any() or (st[1:] <= st[:-1] + ln[:-1] - 1).any() or (st + ln - 1 > HW).any(): bad_run += 1; continue
        if mask_to_rle(rle_to_mask(r)) != r: bad_rt += 1
    chk("empty masks are exactly a single space ' '", bad_fmt == 0, f"{n_empty_} empty rows")
    chk("runs: even count, start >= 1, length >= 1, sorted, non-overlapping, end <= 102400", bad_run == 0, f"{bad_run} bad rows")
    chk("decode -> encode round-trip is exact (1-based, column-major)", bad_rt == 0, f"{bad_rt} mismatches")
    return checks, s
CHECKS, sub_chk = validate_submission(f"{OUT_DIR}/submission.csv", sample_df)
for n, ok, d in CHECKS: print(("✅ " if ok else "❌ ") + n + (f"   [{d}]" if d else ""))
assert all(ok for _, ok, _ in CHECKS), "submission contract violated - do NOT submit"

rate = pd.DataFrame({"test predictions": (sub_chk.mask_rle.str.strip() != "").groupby(sub_chk.defect_type).mean() * 100,
                     "val predictions": [100 * float((PA_rt[:, c] > 0).mean()) for c in range(NC)], "val true labels": [100 * float((GA[:, c] > 0).mean()) for c in range(NC)]}).round(1)
print("\n% of (image, defect) pairs predicted NON-empty:"); display(rate.T)
print("(test firing rate far above validation = shifted / harder test set - explains a leaderboard score below the validation score)")

### Experiment discipline — what is saved so the result can be reproduced and audited
`submission.csv` · `ablation_ladder_m5.csv` · `final_config_m5.json` (members, views, rules, seeds, library versions) · `milestone5_answers.txt/json` · W&B run `m5_final_inference`.
**Rules followed:** one change per ladder row; tuning and scoring on the same images is flagged optimistic and a **cross-fit** number is reported next to it; the sweep engine and the real pipeline are cross-checked on validation before the test set is touched; nothing is tuned on the test set (there are no labels).

In [ ]:
import importlib.metadata as md_
versions = {}
for pkg in ["torch", "torchvision", "numpy", "pandas", "opencv-python", "opencv-python-headless", "scikit-learn", "wandb"]:
    try: versions[pkg] = md_.version(pkg)
    except md_.PackageNotFoundError: pass
cfg = dict(roll_no=ROLL_NO, final=FINAL_NAME, members=FINAL_MEMBERS, views=FINAL_VIEWS, rules={k: (list(map(float, v)) if isinstance(v, list) else v) for k, v in FINAL_RULES.items()},
           split_seed=SEED, val_fraction=VAL_FRACTION, n_val_used=N, pix_mean=PIX_MEAN, pix_std=PIX_STD, strict_gt=True, connectivity=8, rle="1-based column-major, empty = ' '",
           tuned_val_score=float(cand[FINAL_NAME][3]), cross_fit_score=float(cand[FINAL_NAME][2]), test_minutes=round(TEST_MIN, 2), versions=versions)
json.dump(cfg, open(f"{OUT_DIR}/final_config_m5.json", "w"), indent=2)
open(f"{OUT_DIR}/requirements.txt", "w").write("\n".join(f"{k}=={v}" for k, v in versions.items()) + "\n")

ANS["Q9"] = (f"order: read image -> normalise -> model(s) x TTA views (un-flip) -> sigmoid -> average -> strict threshold -> connected-component min-area (+gate) -> RLE (column-major, 1-based) -> rows in sample_submission order; "
             f"final = {FINAL_NAME}; thr {FINAL_RULES['thr']}, min_area {FINAL_RULES['min_area']}, gate {FINAL_RULES['gate']}, mode {FINAL_RULES['mode']}; "
             f"validation tuned {cand[FINAL_NAME][3]:.4f} / cross-fit {cand[FINAL_NAME][2]:.4f}; streamed pipeline == sweep engine (diff {abs(rt['score'] - eng['score']):.5f}); "
             f"integrity checks passed {sum(ok for _, ok, _ in CHECKS)}/{len(CHECKS)}; test inference {TEST_MIN:.1f} min; rows {len(sub_chk)}")
print("ANSWER Q9:", ANS["Q9"])

try:                                                                    # Weights & Biases: leave an auditable record (key from the Kaggle secret WANDB_API_KEY)
    if not USE_WANDB: raise RuntimeError("USE_WANDB = False")
    from kaggle_secrets import UserSecretsClient
    import wandb; wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(project=WANDB_PROJECT, name="m5_final_inference", reinit=True, config=cfg)
    run.log({"ablation_ladder": wandb.Table(dataframe=lad.reset_index().round(5)), "tta_variants": wandb.Table(dataframe=q6.reset_index().round(5)),
             "ensemble_methods": wandb.Table(dataframe=q7.reset_index().astype(str)), "inference_budget": wandb.Table(dataframe=q8.reset_index().astype(str))})
    run.summary.update(dict(val_score_tuned=cand[FINAL_NAME][3], val_score_cross_fit=cand[FINAL_NAME][2], test_minutes=TEST_MIN)); wandb.finish(); print("✅ W&B run logged:", WANDB_PROJECT)
except Exception as e:
    print("⚠️ W&B not logged:", repr(e)[:100], "-> add the secret WANDB_API_KEY (Add-ons -> Secrets) if you want the run recorded")

## ✅ ANSWER SHEET — copy the numbers that match each Google-Form question
Values come from **your** data and models; they depend on the checkpoints attached and can move slightly between runs.

In [ ]:
lines = [f"{q}:  {ANS[q]}" for q in ["Q1", "Q2", "Q3", "Q4", "Q5", "Q6", "Q7", "Q8", "Q9"] if q in ANS]
sheet = "\n\n".join(lines); print("=" * 100 + "\nMILESTONE 5 - ANSWER SHEET\n" + "=" * 100 + "\n\n" + sheet)
open(f"{OUT_DIR}/milestone5_answers.txt", "w").write(sheet); json.dump(ANS, open(f"{OUT_DIR}/milestone5_answers.json", "w"), indent=2)
print("\nfiles in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

### Submit
*Save Version → Save & Run All (Commit)* → Output tab → **Submit to competition** (`submission.csv`).
Viva one-liners: **threshold** = how sure before drawing a pixel (strict `>`; per class); **min-area** = noise filter, costs recall on tiny true defects (Section 7 ceiling); **TTA** = average of flipped views, *each flipped back*; **ensemble** = average probabilities before thresholding; **streaming** = memory independent of the test size; **order of operations** is validated by the validation round-trip before test inference.